In [4]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openpyxl import Workbook
from sklearn.datasets import load_iris
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

FEATURE_MAP = {
    "SepalLengthCm": "sepal length (cm)",
    "SepalWidthCm": "sepal width (cm)",
    "PetalLengthCm": "petal length (cm)",
    "PetalWidthCm": "petal width (cm)",
}


def load_iris_dataframe() -> pd.DataFrame:
    iris = load_iris(as_frame=True)
    df = iris.frame.copy()
    df.rename(
        columns={
            "sepal length (cm)": "SepalLengthCm",
            "sepal width (cm)": "SepalWidthCm",
            "petal length (cm)": "PetalLengthCm",
            "petal width (cm)": "PetalWidthCm",
        },
        inplace=True,
    )
    species_name = pd.Categorical.from_codes(df["target"], iris.target_names)
    df["Species"] = "Iris-" + species_name.astype(str)
    return df.drop(columns=["target"])


def grouped_summary(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby("Species")[list(FEATURE_MAP.keys())].describe().round(3)


def fit_simple_linear_regressions(df: pd.DataFrame) -> pd.DataFrame:
    records: List[Dict[str, object]] = []
    features = list(FEATURE_MAP.keys())

    for y in features:
        for x in features:
            if x == y:
                continue
            X = df[[x]].values
            y_true = df[y].values
            model = LinearRegression().fit(X, y_true)
            y_hat = model.predict(X)
            residuals = y_true - y_hat
            records.append(
                {
                    "Pair": f"{x.replace('Cm','')} x {y.replace('Cm','')}",
                    "X": x,
                    "Y": y,
                    "coef": model.coef_[0],
                    "intercept": model.intercept_,
                    "MSE": mean_squared_error(y_true, y_hat),
                    "R2": r2_score(y_true, y_hat),
                    "ResidualMean": residuals.mean(),
                    "ResidualStd": residuals.std(ddof=1),
                }
            )

    return pd.DataFrame(records).sort_values(["Y", "X"]).reset_index(drop=True)


def fit_multiple_linear_regressions(df: pd.DataFrame) -> pd.DataFrame:
    records: List[Dict[str, object]] = []
    features = list(FEATURE_MAP.keys())

    for y in features:
        x_cols = [c for c in features if c != y]
        X = df[x_cols].values
        y_true = df[y].values
        model = LinearRegression().fit(X, y_true)
        y_hat = model.predict(X)
        records.append(
            {
                "Target": y,
                "Predictors": ", ".join(x_cols),
                "MSE": mean_squared_error(y_true, y_hat),
                "R2": r2_score(y_true, y_hat),
                "Intercept": model.intercept_,
                "Coef": model.coef_.tolist(),
            }
        )

    return pd.DataFrame(records).sort_values("Target").reset_index(drop=True)


def polynomial_ridge_search(
    df: pd.DataFrame,
    target: str = "SepalLengthCm",
    degrees: Tuple[int, ...] = (1, 3, 5),
    alphas: Tuple[float, ...] = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0),
) -> Tuple[pd.DataFrame, GridSearchCV, pd.DataFrame]:
    x_cols = [c for c in FEATURE_MAP if c != target]
    X = df[x_cols].values
    y = df[target].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    pipe = Pipeline(
        steps=[
            ("poly", PolynomialFeatures(include_bias=False)),
            ("scaler", StandardScaler()),
            ("ridge", Ridge()),
        ]
    )

    params = {
        "poly__degree": list(degrees),
        "ridge__alpha": list(alphas),
    }

    search = GridSearchCV(
        estimator=pipe,
        param_grid=params,
        scoring="neg_mean_squared_error",
        cv=5,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)

    best = search.best_estimator_
    y_hat_train = best.predict(X_train)
    y_hat_test = best.predict(X_test)

    summary = pd.DataFrame(
        [
            {
                "target": target,
                "predictors": ", ".join(x_cols),
                "best_degree": search.best_params_["poly__degree"],
                "best_alpha": search.best_params_["ridge__alpha"],
                "train_mse": mean_squared_error(y_train, y_hat_train),
                "test_mse": mean_squared_error(y_test, y_hat_test),
                "train_r2": r2_score(y_train, y_hat_train),
                "test_r2": r2_score(y_test, y_hat_test),
            }
        ]
    )

    cv_df = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
    return summary, search, cv_df


def make_slr_plots(df: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    features = list(FEATURE_MAP.keys())

    for y in features:
        for x in features:
            if x == y:
                continue
            X = df[[x]].values
            y_true = df[y].values
            model = LinearRegression().fit(X, y_true)
            y_hat = model.predict(X)

            plt.figure(figsize=(6, 4))
            plt.scatter(df[x], y_true, s=18, alpha=0.75)
            order = np.argsort(df[x].values)
            plt.plot(df[x].values[order], y_hat[order], linewidth=2)
            plt.xlabel(x)
            plt.ylabel(y)
            plt.title(f"SLR: {x} -> {y}")
            plt.tight_layout()
            plt.savefig(out_dir / f"slr_{x}_to_{y}.png", dpi=150)
            plt.close()


def export_excel_layout(
    slr_df: pd.DataFrame,
    mlr_df: pd.DataFrame,
    out_path: Path,
) -> None:
    """Export SLR/MLR results to an Excel file in a table-oriented layout."""
    wb = Workbook()
    ws_slr = wb.active
    ws_slr.title = "SLR-1"
    ws_mlr = wb.create_sheet("MLR-1")

    # -------- SLR sheet layout --------
    ws_slr["A1"] = "List"
    ws_slr["B1"] = "RegPlot (ŷ)"
    ws_slr["C1"] = "Residual Check"
    ws_slr["D1"] = "MSE"
    ws_slr["E1"] = "R-square"

    for idx, row in enumerate(slr_df.itertuples(index=False), start=2):
        ws_slr[f"A{idx}"] = f"{row.X.replace('Cm','')} x {row.Y.replace('Cm','')}"
        ws_slr[f"B{idx}"] = f"ŷ = {row.intercept:.4f} + ({row.coef:.4f})*x"
        ws_slr[f"C{idx}"] = f"mean={row.ResidualMean:.4f}, std={row.ResidualStd:.4f}"
        ws_slr[f"D{idx}"] = float(row.MSE)
        ws_slr[f"E{idx}"] = float(row.R2)

    # -------- MLR sheet layout --------
    ws_mlr["A1"] = "Target (Y)"
    ws_mlr["B1"] = "Predictors (X)"
    ws_mlr["C1"] = "Equation (ŷ)"
    ws_mlr["D1"] = "MSE"
    ws_mlr["E1"] = "R-square"

    for idx, row in enumerate(mlr_df.itertuples(index=False), start=2):
        predictors = row.Predictors.split(", ")
        coefs = list(row.Coef)
        terms = " + ".join([f"({coef:.4f})*{pred}" for coef, pred in zip(coefs, predictors)])
        ws_mlr[f"A{idx}"] = row.Target
        ws_mlr[f"B{idx}"] = row.Predictors
        ws_mlr[f"C{idx}"] = f"ŷ = {row.Intercept:.4f} + {terms}"
        ws_mlr[f"D{idx}"] = float(row.MSE)
        ws_mlr[f"E{idx}"] = float(row.R2)

    # set friendly column widths
    for ws in (ws_slr, ws_mlr):
        ws.column_dimensions["A"].width = 24
        ws.column_dimensions["B"].width = 42
        ws.column_dimensions["C"].width = 38
        ws.column_dimensions["D"].width = 16
        ws.column_dimensions["E"].width = 16

    wb.save(out_path)


def main() -> None:
    df = load_iris_dataframe()

    out = Path("outputs")
    out.mkdir(exist_ok=True)

    grouped_summary(df).to_csv(out / "grouped_summary.csv")

    slr = fit_simple_linear_regressions(df)
    slr.to_csv(out / "slr_results.csv", index=False)

    mlr = fit_multiple_linear_regressions(df)
    mlr.to_csv(out / "mlr_results.csv", index=False)
    export_excel_layout(slr, mlr, out / "iris_slr_mlr_layout.xlsx")

    ridge_summary, _, cv_results = polynomial_ridge_search(df)
    ridge_summary.to_csv(out / "poly_ridge_best_model.csv", index=False)
    cv_results.to_csv(out / "poly_ridge_cv_results.csv", index=False)

    make_slr_plots(df, out / "slr_plots")

    print("Done. Files created under ./outputs")


if __name__ == "__main__":
    main()

Done. Files created under ./outputs
